<a href="https://colab.research.google.com/github/marcouras/AI-engineering-fundamentals/blob/main/lezione4/Lezione4_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

# 🤖 AI Engineering Fundamentals
## Lezione 4 — RAG: Conoscenza Personalizzata

**ITS Novitas 4.0 — Sviluppatore Intelligenza Artificiale**  
Docente: Marco Uras | 📅 Giovedì 28/05/2026

---

### 🎯 Obiettivi
- ✅ Capire la pipeline RAG completa
- ✅ Indicizzare un PDF con ChromaDB
- ✅ Implementare la ricerca semantica
- ✅ Integrare RAG nel chatbot esistente

In [1]:
# Setup — eseguite questa cella per prima
import anthropic, os
from dotenv import load_dotenv

# Load variables from .env file
load_dotenv("../")

# Retrieve the API key
key = os.getenv('ANTHROPIC_API_KEY')

client = anthropic.Anthropic(api_key=key)

def chiedi_claude(domanda, system=None, max_tokens=800):
    params = {"model":"claude-haiku-4-5-20251001","max_tokens":max_tokens,
              "messages":[{"role":"user","content":domanda}]}
    if system: params["system"] = system
    return client.messages.create(**params).content[0].text

print("✅ Setup completato!")

✅ Setup completato!


---
## 1. Crea un documento di test

Per l'esercizio creiamo un documento di testo su WiData. In un progetto reale useresti un PDF vero.

In [2]:
# Creiamo un documento di testo su WiData
documento_widata = """
WiData Srl — Manuale Prodotti IoT

SENSORE XS200 - MONITORAGGIO AMBIENTALE
Il sensore XS200 è progettato per il monitoraggio ambientale in ambienti industriali e urbani.
Misura temperatura (-20°C a +60°C), umidità relativa (0-100%), pressione atmosferica e qualità dell'aria (CO2, PM2.5).
Classificazione IP67: impermeabile e resistente alla polvere. Alimentazione: batteria Li-Ion 3.7V, autonomia 2 anni.
Connettività: LoRaWAN, NB-IoT, WiFi 802.11n. Dimensioni: 85x45x30mm. Peso: 120g.
Certificazioni: CE, FCC, RoHS. Garanzia: 3 anni.

GATEWAY GW500 - CONCENTRATORE DATI
Il gateway GW500 raccoglie dati da fino a 1000 sensori simultaneamente tramite LoRaWAN.
Copertura fino a 15km in aree rurali, 3km in aree urbane. Elaborazione edge computing integrata.
Connessione cloud via Ethernet, WiFi o 4G LTE. Storage locale: 32GB SSD.
Alimentazione: 220V AC o pannello solare (opzionale). Temperatura operativa: -40°C a +70°C.
Certificazioni: CE, IP65. Installazione: palo, tetto o rack.

PIATTAFORMA XPLORE - ANALYTICS
Xplore è la piattaforma cloud di WiData per la visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-time, storico dati fino a 5 anni.
Alerting automatico via email, SMS o webhook quando i valori superano soglie configurabili.
API REST per integrazione con sistemi terzi (ERP, SCADA, BIM).
Machine learning per previsione anomalie e manutenzione predittiva.
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiesta).
SLA: 99.9% uptime garantito nei piani Pro ed Enterprise.

SUPPORTO E ASSISTENZA
Supporto tecnico disponibile lunedì-venerdì 9:00-18:00.
Email: support@widata.cloud | Telefono: +39 079 123456.
Per informazioni commerciali: sales@widata.cloud.
Sede: Via Roma 42, Sassari (SS) 07100, Italia.
Spedizioni in tutta Italia in 3-5 giorni lavorativi.
"""

# Salva su file
with open("manuale_widata.txt", "w", encoding="utf-8") as f:
    f.write(documento_widata)

print(f"✅ Documento creato: {len(documento_widata)} caratteri")

✅ Documento creato: 1846 caratteri


---
## 2. Chunking e Indicizzazione

In [3]:
def chunka_testo(testo, chunk_size=400, overlap=50):
    """Divide il testo in chunk con overlap."""
    chunks = []
    start = 0
    while start < len(testo):
        end = start + chunk_size
        chunk = testo[start:end]
        if chunk.strip():  # ignora chunk vuoti
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

chunks = chunka_testo(documento_widata)
print(f"📊 Numero di chunk: {len(chunks)}")
print()
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ({len(chunk)} char) ---")
    print(chunk[:100]+"...")
    print()

📊 Numero di chunk: 6

--- Chunk 1 (400 char) ---

WiData Srl — Manuale Prodotti IoT

SENSORE XS200 - MONITORAGGIO AMBIENTALE
Il sensore XS200 è proge...

--- Chunk 2 (400 char) ---
. Alimentazione: batteria Li-Ion 3.7V, autonomia 2 anni.
Connettività: LoRaWAN, NB-IoT, WiFi 802.11n...

--- Chunk 3 (400 char) ---
km in aree urbane. Elaborazione edge computing integrata.
Connessione cloud via Ethernet, WiFi o 4G ...

--- Chunk 4 (400 char) ---
iData per la visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-tim...

--- Chunk 5 (400 char) ---
va.
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiest...

--- Chunk 6 (96 char) ---
: Via Roma 42, Sassari (SS) 07100, Italia.
Spedizioni in tutta Italia in 3-5 giorni lavorativi.
...



In [4]:
import chromadb

# Crea il client ChromaDB in memoria
chroma_client = chromadb.Client()

# Crea o recupera la collection
collection = chroma_client.get_or_create_collection(
    name="widata_docs",
    metadata={"hnsw:space": "cosine"}  # usa similarità coseno
)

# Indicizza i chunk
collection.add(
    documents=chunks,
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)

print(f"✅ Indicizzati {collection.count()} chunk in ChromaDB")
print("💡 ChromaDB ha calcolato automaticamente gli embedding per ogni chunk!")

✅ Indicizzati 6 chunk in ChromaDB
💡 ChromaDB ha calcolato automaticamente gli embedding per ogni chunk!


---
## 3. Ricerca Semantica

In [23]:
def cerca(domanda, collection, n_risultati=3):
    """Cerca i chunk più rilevanti per la domanda."""
    risultati = collection.query(
        query_texts=[domanda],
        n_results=n_risultati
    )
    return risultati["documents"][0]

# Test ricerca semantica
domande_test = [
    "Quali sensori supportate per ambienti esterni?",
    "Come posso integrare i dati con il mio sistema ERP?",
    "Qual è il costo del piano professionale?",
]

for domanda in domande_test:
    print(f"\n❓ {domanda}")
    chunks_trovati = cerca(domanda, collection, n_risultati=2)
    for i, chunk in enumerate(chunks_trovati):
        print(f"  📄 Chunk {i+1}: {chunk[:120]}...")


❓ Quali sensori supportate per ambienti esterni?
  📄 Chunk 1: va.
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiesta).
SLA: 99.9% uptim...
  📄 Chunk 2: iData per la visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-time, storico dati fino...

❓ Come posso integrare i dati con il mio sistema ERP?
  📄 Chunk 1: va.
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiesta).
SLA: 99.9% uptim...
  📄 Chunk 2: iData per la visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-time, storico dati fino...

❓ Qual è il costo del piano professionale?
  📄 Chunk 1: : Via Roma 42, Sassari (SS) 07100, Italia.
Spedizioni in tutta Italia in 3-5 giorni lavorativi.
...
  📄 Chunk 2: va.
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiesta).
SLA: 99.9% uptim...


---
## 4. RAG Completo — Domanda + Contesto + Risposta

In [6]:
SYSTEM_WIDATA = """
Sei l'assistente virtuale di WiData Srl, azienda IoT e smart cities di Sassari.
Rispondi SOLO basandoti sui documenti forniti nel contesto.
Se la risposta non è nei documenti, dì chiaramente: 'Non ho questa informazione nei miei documenti.'
Non inventare mai informazioni. Sii conciso e preciso.
"""

def chat_rag(domanda, collection, n_chunks=3):
    """Chatbot con RAG: recupera contesto e genera risposta."""
    # 1. Recupera i chunk rilevanti
    chunks_rilevanti = cerca(domanda, collection, n_risultati=n_chunks)
    contesto = "\n\n---\n\n".join(chunks_rilevanti)

    # 2. Costruisci il prompt aumentato
    prompt = f"""Documenti di riferimento:

{contesto}

---

Domanda dell'utente: {domanda}"""

    # 3. Genera la risposta
    risposta = chiedi_claude(prompt, system=SYSTEM_WIDATA)
    return risposta, chunks_rilevanti

# Test completo
domanda = "Il sensore XS200 funziona in ambienti molto freddi?"
risposta, chunks = chat_rag(domanda, collection)

print(f"❓ {domanda}")
print(f"\n🤖 {risposta}")
print(f"\n📄 Basato su {len(chunks)} chunk")

❓ Il sensore XS200 funziona in ambienti molto freddi?

🤖 Sì, il sensore XS200 funziona in ambienti freddi. Secondo le specifiche tecniche, ha un **range di temperatura operativa da -20°C a +60°C**.

Quindi può operare correttamente fino a temperature di -20°C, rendendolo adatto anche per ambienti molto freddi.

📄 Basato su 3 chunk


In [7]:
# Test con domanda fuori dai documenti
domanda_off = "Quali sono i migliori smartphone del 2025?"
risposta_off, _ = chat_rag(domanda_off, collection)
print(f"❓ {domanda_off}")
print(f"\n🤖 {risposta_off}")
print("\n💡 Il sistema dovrebbe rifiutarsi di rispondere!")

❓ Quali sono i migliori smartphone del 2025?

🤖 Non ho questa informazione nei miei documenti.

I documenti che ho a disposizione riguardano i prodotti e servizi di WiData Srl (sensori IoT, gateway, piattaforma analytics e piani tariffari), non smartphone.

Posso aiutarti con informazioni su soluzioni IoT e smart cities di WiData?

💡 Il sistema dovrebbe rifiutarsi di rispondere!


---
## ⭐ Esercizi

In [8]:
NOME_STUDENTE = "Alfonso Mammato"  # ← SCRIVI IL TUO NOME
if NOME_STUDENTE:
    print(f"✅ Notebook di: {NOME_STUDENTE}")
else:
    print("⚠️ Scrivi il tuo nome!")

✅ Notebook di: Alfonso Mammato


### Esercizio 1 — Indicizza un documento tuo ★☆☆
Crea un documento di testo su un argomento a tua scelta (può essere anche una dispensa del corso, una ricetta, un regolamento). Indicizzalo in ChromaDB e fai 3 domande. I chunk recuperati sono rilevanti?

In [24]:
# ESERCIZIO 1
mio_documento = """
# Sistema di Gestione della Conoscenza Aziendale

L'obiettivo di questo progetto è realizzare una piattaforma centralizzata per la gestione e la consultazione delle informazioni aziendali. Il sistema consente ai dipendenti di accedere rapidamente a documentazione tecnica, procedure operative, policy interne e materiale formativo attraverso un'interfaccia di ricerca intelligente.

La piattaforma raccoglie contenuti provenienti da diverse fonti, tra cui repository documentali, wiki aziendali, database relazionali e sistemi di ticketing. Tutti i documenti vengono indicizzati periodicamente per garantire la disponibilità delle informazioni più aggiornate.

Gli utenti possono effettuare ricerche utilizzando parole chiave o formulando domande in linguaggio naturale. Il motore di ricerca individua i documenti più rilevanti in base al contenuto e restituisce una selezione delle fonti pertinenti. Successivamente, un componente di generazione automatica produce una risposta sintetica basata esclusivamente sulle informazioni recuperate.

Tra i principali vantaggi della soluzione vi sono la riduzione dei tempi di ricerca delle informazioni, il miglioramento della produttività e la diminuzione del rischio di utilizzare dati obsoleti o non verificati. Il sistema mantiene inoltre la tracciabilità delle fonti utilizzate per generare le risposte, consentendo agli utenti di verificare facilmente l'origine delle informazioni.

La piattaforma è progettata per supportare elevati volumi di documenti e accessi simultanei. L'architettura prevede componenti modulari per l'ingestione dei dati, l'indicizzazione, il recupero delle informazioni e la generazione delle risposte. Questo approccio facilita l'integrazione con sistemi esistenti e permette di estendere facilmente le funzionalità nel tempo.

Il progetto rappresenta un esempio concreto di applicazione dell'intelligenza artificiale generativa in ambito enterprise, con particolare attenzione alla qualità delle risposte, alla governance dei dati e alla sicurezza delle informazioni aziendali.
"""


# Crea una nuova collection
mia_collection = chroma_client.get_or_create_collection(
    name="mio_documento",
    metadata={"hnsw:space": "cosine"}  # usa similarità coseno
)

# Chunka e indicizza
chunks = chunka_testo(mio_documento)
mia_collection.add(
    documents=chunks,
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)

# Fai 3 domande e stampa i chunk recuperati
domande_test = [
    "Qual è l'obiettivo del progetto?",
    "Cosa possono fare gli utenti?",
    "Per cosa è progettata la piattaforma?",
]

for domanda in domande_test:
    print(f"\n❓ {domanda}")
    chunks_trovati = cerca(domanda, mia_collection, n_risultati=1)
    for i, chunk in enumerate(chunks_trovati):
        print(f"  📄 Chunk {i+1}: {chunk[:120]}...")


❓ Qual è l'obiettivo del progetto?
  📄 Chunk 1: e, consentendo agli utenti di verificare facilmente l'origine delle informazioni.

La piattaforma è progettata per suppo...

❓ Cosa possono fare gli utenti?
  📄 Chunk 1: rfaccia di ricerca intelligente.

La piattaforma raccoglie contenuti provenienti da diverse fonti, tra cui repository do...

❓ Per cosa è progettata la piattaforma?
  📄 Chunk 1: 
# Sistema di Gestione della Conoscenza Aziendale

L'obiettivo di questo progetto è realizzare una piattaforma centraliz...


### Esercizio 2 — Sperimenta con il chunking ★★☆
Prova a indicizzare lo stesso documento con chunk_size=200, 400 e 800. Per la stessa domanda, i chunk recuperati sono diversi? Quale dimensione dà risultati migliori?

In [27]:
# ESERCIZIO 2
domanda_test = "Per cosa è progettata la piattaforma?"  # ← modifica
print(domanda_test)

for chunk_size in [200, 400, 2000]:
    print(f"\n{'='*50}")
    print(f"chunk_size = {chunk_size}")
    print('='*50)

    # TODO: crea collection, chunka con dimensione diversa, indicizza, cerca
    chunks = chunka_testo(mio_documento, chunk_size=chunk_size)
    mia_collection.add(
        documents=chunks,
        ids=[f"chunk_{i}" for i in range(len(chunks))]
    )
    risultati = cerca(domanda_test, mia_collection)
    for i, chunk in enumerate(risultati):
        print(f"  📄 Chunk {i+1}: {chunk}...")
    
    pass

# Commento: quale chunk_size ha dato i risultati migliori?
# Risposta: ...

Per cosa è progettata la piattaforma?

chunk_size = 200
  📄 Chunk 1: 
# Sistema di Gestione della Conoscenza Aziendale

L'obiettivo di questo progetto è realizzare una piattaforma centralizzata per la gestione e la consultazione delle informazioni aziendali. Il sistema consente ai dipendenti di accedere rapidamente a documentazione tecnica, procedure operative, policy interne e materiale formativo attraverso un'interfaccia di ricerca intelligente.

La piattaforma r...
  📄 Chunk 2: e, consentendo agli utenti di verificare facilmente l'origine delle informazioni.

La piattaforma è progettata per supportare elevati volumi di documenti e accessi simultanei. L'architettura prevede c...
  📄 Chunk 3: e l'origine delle informazioni.

La piattaforma è progettata per supportare elevati volumi di documenti e accessi simultanei. L'architettura prevede componenti modulari per l'ingestione dei dati, l'indicizzazione, il recupero delle informazioni e la generazione delle risposte. Questo approccio fa

### Esercizio 3 — RAG + storia conversazione ★★☆
Integra RAG nella funzione `chat()` della Lezione 3 (quella con la history). Ogni risposta deve usare sia il contesto RAG che la storia della conversazione.

In [31]:
# ESERCIZIO 3
import json, os

MEMORY_FILE = "chatbot_widata.json"
MAX_MESSAGGI = 4

def carica_storia():
    """Carica la history dal file JSON. Restituisce lista vuota se non esiste."""
    if os.path.exists(MEMORY_FILE):
        with open(MEMORY_FILE, "r", encoding="utf-8") as f:
            storia = json.load(f)
            print(f"📂 Storia caricata: {len(storia)} messaggi precedenti")
            return storia
    print("🆕 Nessuna storia precedente — nuova conversazione")
    return []

def salva_storia(history):
    """Salva la history su file JSON."""
    with open(MEMORY_FILE, "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)
    print(f"💾 Storia salvata: {len(history)} messaggi")


def chat_rag_con_storia(domanda, collection, history):
    """Chatbot con RAG + conversazione multi-turno."""
    # TODO:

    # 1. Recupera chunk rilevanti con cerca()
    chunks_rilevanti = cerca(domanda, collection)

    # 2. Costruisci il messaggio con contesto + domanda
    contesto = "\n\n---\n\n".join(chunks_rilevanti)
    prompt = f"""Documenti di riferimento:

    {contesto}

    ---

    Domanda dell'utente: {domanda}"""

    # 3. Aggiungi alla history
    history.append({"role": "user", "content": prompt})

    # 4. Chiama l'API con tutta la history
        # è chiamata nel loop

    # 5. Aggiungi la risposta alla history
    full_text = ""

    with client.messages.stream(
        model="claude-haiku-4-5-20251001",
        max_tokens=300,
        messages=history,
        system=SYSTEM_WIDATA,
    ) as stream:

        for text in stream.text_stream:
            print(text, end="", flush=True)
            full_text += text

        # Messaggio finale completo con usage
        final_message = stream.get_final_message()

    print()

    token_input = final_message.usage.input_tokens
    token_output = final_message.usage.output_tokens
    print(f"Totale token: {token_input + token_output}")

    history.append({"role": "assistant", "content": full_text})

    # 6. Restituisci la risposta
        # viene printata nello stream
    pass

# Test: domande collegate che richiedono sia RAG che memoria
# chat_rag_con_storia("Parlami del sensore XS200")
# chat_rag_con_storia("Qual è la sua autonomia?")
# chat_rag_con_storia("Costa molto?")

# Loop principale
def main():
    history = carica_storia()
    print("🤖 Chatbot WiData avviato. Digita 'esci' per uscire.\n")

    while True:
        utente = input("Tu: ")
        if utente.lower() == "esci":
            print("👋 Arrivederci!")
            break

        # Lo streaming stampa già durante l'esecuzione
        chat_rag_con_storia(utente, collection, history)

    salva_storia(history)

# Esecuzione
main()  # Decommentare per eseguire

📂 Storia caricata: 0 messaggi precedenti
🤖 Chatbot WiData avviato. Digita 'esci' per uscire.

# Sensore XS200 - Monitoraggio Ambientale

Il sensore **XS200** è progettato per il monitoraggio ambientale in ambienti industriali e urbani.

## Caratteristiche tecniche:

**Parametri misurati:**
- Temperatura: -20°C a +60°C
- Umidità relativa: 0-100%
- Pressione atmosferica
- Qualità dell'aria (CO2, PM2.5)

**Robustezza:**
- Classificazione IP67: impermeabile e resistente alla polvere

**Alimentazione:**
- Batteria Li-Ion 3.7V
- Autonomia: 2 anni (il documento risulta troncato, ma indica un'autonomia significativa)

È una soluzione ideale per applicazioni di monitoraggio ambientale che richiedono resistenza agli agenti atmosferici e affidabilità nel tempo.
Totale token: 794
In base ai documenti forniti, non ho informazioni complete sull'autonomia del sensore XS200.

Dal precedente documento risultava che il sensore XS200 ha una **batteria Li-Ion 3.7V con autonomia di 2 anni**, ma il testo er

### Esercizio 4 — Chatbot RAG WiData completo ★★★ (Deliverable!)

Costruisci il chatbot completo con:
- RAG sul documento WiData
- Conversation history (sliding window)
- Streaming
- System prompt WiData con istruzione anti-hallucination
- Loop interattivo con `input()`
- Stampa i chunk usati per ogni risposta (per debug)

In [35]:
# ESERCIZIO 4 — Chatbot RAG completo (DELIVERABLE)
import chromadb, json, anthropic, os
from dotenv import load_dotenv

# Load variables from .env file
load_dotenv("../")

# Retrieve the API key
key = os.getenv('ANTHROPIC_API_KEY')
client = anthropic.Anthropic(api_key=key)

SYSTEM_FILE = "widata_system.txt"
MEMORY_FILE = "chatbot_widata.json"
DOCUMENT_FILE = "manuale_widata.txt"
MAX_MESSAGGI = 10

def carica_storia():
    """Carica la history dal file JSON. Restituisce lista vuota se non esiste."""
    if os.path.exists(MEMORY_FILE):
        with open(MEMORY_FILE, "r", encoding="utf-8") as f:
            storia = json.load(f)
            print(f"📂 Storia caricata: {len(storia)} messaggi precedenti")
            return storia
    print("🆕 Nessuna storia precedente — nuova conversazione")
    return []

def salva_storia(history):
    """Salva la history su file JSON."""
    with open(MEMORY_FILE, "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)
    print(f"💾 Storia salvata: {len(history)} messaggi")

def carica_system():
    """Carica system prompt da file txt"""
    
    if os.path.exists(SYSTEM_FILE):
        with open(SYSTEM_FILE, "r", encoding="utf-8") as f:
            system_file = f.read()
            print("System file caricato")
            return system_file

    print("Nessun system file trovato")

    system_file = """
Sei l'assistente virtuale di WiData Srl, azienda IoT e smart cities di Sassari.
Rispondi SOLO basandoti sui documenti forniti nel contesto.
Se la risposta non è nei documenti, dì chiaramente: 'Non ho questa informazione nei miei documenti.'
Non inventare mai informazioni. Sii conciso e preciso.
"""

    return system_file

SYSTEM = carica_system()

def carica_rag_data():
    """Carica fonte dei dati per la rag"""
    if os.path.exists(DOCUMENT_FILE):
        with open(DOCUMENT_FILE, "r", encoding="utf-8") as f:
            document_file = f.read()
            print(f"Fonti dati per il RAG caricate.")
            return document_file
    print("Nessun system file trovato")
    return "Nessun documento utilizzabile per il RAG"

RAG_DATA = carica_rag_data()

def setup_rag(testo, chunk_size=400, overlap=50, name="widata_docs"):
    """Indicizza il documento e restituisce la collection."""

    chunks = []
    start = 0
    while start < len(testo):
        end = start + chunk_size
        chunk = testo[start:end]
        if chunk.strip():  # ignora chunk vuoti
            chunks.append(chunk)
        start += chunk_size - overlap

    # Crea il client ChromaDB in memoria
    chroma_client = chromadb.Client()

    # Crea o recupera la collection
    collection = chroma_client.get_or_create_collection(
        name=name,
        metadata={"hnsw:space": "cosine"}  # usa similarità coseno
    )

    # Indicizza i chunk
    collection.add(
        documents=chunks,
        ids=[f"chunk_{i}" for i in range(len(chunks))]
    )

    return collection

def cerca(domanda, collection, n_risultati=3):
    """Ricerca semantica nella collection.
        Cerca i chunk più rilevanti per la domanda."""
    
    risultati = collection.query(
        query_texts=[domanda],
        n_results=n_risultati
    )
    return risultati["documents"][0]

def chat_completo(domanda, history, collection, max_messaggi=MAX_MESSAGGI):
    """Chatbot con RAG + storia + streaming."""
    # 1. Recupera chunk rilevanti con cerca()
    chunks_rilevanti = cerca(domanda, collection, n_risultati=1)

    # 2. Costruisci il messaggio con contesto + domanda
    contesto = "\n\n---\n\n".join(chunks_rilevanti)
    prompt = f"""Documenti di riferimento:

    {contesto}

    ---

    Domanda dell'utente: {domanda}"""

    # 3. Aggiungi alla history
    history.append({"role": "user", "content": prompt})

    # 4. Chiama l'API con tutta la history
        # è chiamata nel loop

    # 5. Aggiungi la risposta alla history
    full_text = ""

    with client.messages.stream(
        model="claude-haiku-4-5-20251001",
        max_tokens=300,
        messages=history,
        system=SYSTEM_WIDATA,
    ) as stream:

        for text in stream.text_stream:
            print(text, end="", flush=True)
            full_text += text

        # Messaggio finale completo con usage
        final_message = stream.get_final_message()

    print()

    token_input = final_message.usage.input_tokens
    token_output = final_message.usage.output_tokens
    print(f"Totale token: {token_input + token_output}")

    history.append({"role": "assistant", "content": full_text})


    # CONTEXT WINDOW
    history[:] = history[-max_messaggi * 2:]

    # 6. Restituisci la risposta
        # viene printata nello stream
    pass


def main():
    collection = setup_rag(RAG_DATA)
    history = carica_storia()

    print("🤖 Chatbot WiData RAG avviato. Digita 'esci' per uscire.\n")

    while True:
        utente = input("Tu: ")
        if utente.lower() == "esci":
            print("👋 Arrivederci!")
            break
        chat_completo(utente, history, collection)
    
    salva_storia(history)


main()  # Decommentare per eseguire

# Test: domande collegate che richiedono sia RAG che memoria
# chat_rag_con_storia("Parlami del sensore XS200")
# chat_rag_con_storia("Qual è la sua autonomia?")
# chat_rag_con_storia("Costa molto?")


System file caricato
Fonti dati per il RAG caricate.
📂 Storia caricata: 14 messaggi precedenti
🤖 Chatbot WiData RAG avviato. Digita 'esci' per uscire.

Non ho questa informazione nei miei documenti.

I documenti a mia disposizione contengono informazioni su:
- Prodotti e servizi IoT di WiData
- Piani di abbonamento e prezzi
- Contatti e sede dell'azienda
- Supporto tecnico

Per informazioni su persone specifiche, ti consiglio di contattare direttamente WiData:

📧 **sales@widata.cloud** | 📞 **+39 079 123456**  
📍 Via Roma 42, Sassari (SS) 07100, Italia
Totale token: 4898
👋 Arrivederci!
💾 Storia salvata: 16 messaggi


---
## 📤 Consegna

1. Completa tutti gli esercizi
2. Scarica: `File → Scarica → .ipynb`
3. Rinomina: `Lezione4_TUONOME.ipynb`
4. Carica su GitHub in `lezione4/`

```bash
git add lezione4/
git commit -m "Lezione 4 completata"
git push
```

---
### 📖 Per la prossima lezione (Giovedì 04/06)
Leggi **Huyen Cap. 6 — sezione Agents**

---
*ITS Novitas 4.0 — AI Engineering Fundamentals | Marco Uras*